# Taller NetLogo + R: efecto de la cuarentena en la propagación de una infección

**Objetivo:** analizar los resultados exportados desde BehaviorSpace (NetLogo) y comparar cómo el porcentaje de personas en cuarentena cambia la velocidad del contagio.

**Antes de empezar:** en Colab use *Entorno de ejecución → Cambiar tipo de entorno de ejecución → R*, y suba a la carpeta `/content` el CSV exportado de NetLogo en formato **Table** (no Spreadsheet).

In [ ]:
library(tidyverse)
options(repr.plot.width = 9, repr.plot.height = 5)

## 1. Cargar los datos
El CSV de NetLogo trae unas líneas de metadatos antes de la tabla real. Buscamos automáticamente la fila del encabezado.

In [ ]:
archivo <- "/content/cuarentena-table.csv"   # cambie el nombre si es distinto

lineas <- readLines(archivo)
inicio <- grep("\\[run number\\]", lineas)[1]
datos  <- read.csv(archivo, skip = inicio - 1)
head(datos)

## 2. Limpiar y renombrar columnas

In [ ]:
datos <- datos %>%
  rename(corrida    = starts_with("X.run"),
         cuarentena = starts_with("porcentaje"),
         personas   = starts_with("num"),
         radio      = starts_with("radio"),
         tick       = starts_with("X.step"),
         infectados = starts_with("count"))

glimpse(datos)

## 3. Tiempo hasta infectar a toda la población
Para cada corrida calculamos el primer *tick* en que todos están infectados (`NA` si nunca ocurrió dentro del límite de 500 ticks).

In [ ]:
tiempos <- datos %>%
  group_by(corrida, cuarentena) %>%
  summarise(
    tiempo = if_else(any(infectados >= personas),
                     min(tick[infectados >= personas]), NA_real_),
    .groups = "drop")

resumen <- tiempos %>%
  group_by(cuarentena) %>%
  summarise(promedio   = mean(tiempo, na.rm = TRUE),
            desviacion = sd(tiempo, na.rm = TRUE),
            completas  = sum(!is.na(tiempo)),
            corridas   = n())
resumen

## 4. Serie de tiempo: infectados promedio por nivel de cuarentena

In [ ]:
promedio_tick <- datos %>%
  group_by(cuarentena, tick) %>%
  summarise(infectados = mean(infectados), .groups = "drop")

ggplot(promedio_tick, aes(tick, infectados, color = factor(cuarentena))) +
  geom_line(linewidth = 1) +
  labs(title = "Propagación de la infección según % de cuarentena",
       x = "Ticks", y = "Infectados (promedio)", color = "% cuarentena") +
  theme_minimal()

## 5. Tiempo promedio de infección con barras de error (± 1 desviación estándar)

In [ ]:
resumen %>%
  filter(!is.na(promedio)) %>%
  ggplot(aes(cuarentena, promedio)) +
  geom_line() +
  geom_point(size = 3) +
  geom_errorbar(aes(ymin = promedio - desviacion, ymax = promedio + desviacion), width = 2) +
  labs(title = "A mayor cuarentena, más tarde se infecta toda la población",
       x = "% de personas en cuarentena", y = "Ticks hasta infección total") +
  theme_minimal()

## 6. Curvas de tendencia suavizadas

In [ ]:
ggplot(datos, aes(tick, infectados, color = factor(cuarentena))) +
  geom_smooth(se = FALSE) +
  labs(title = "Curvas de tendencia por nivel de cuarentena",
       x = "Ticks", y = "Infectados", color = "% cuarentena") +
  theme_minimal()

## 7. Conclusiones
*(Escriba aquí sus conclusiones: ¿cómo cambia el tiempo de infección al aumentar la cuarentena? ¿Qué pasa con 100 % de cuarentena? ¿Qué limitaciones tiene el modelo?)*